# J2 — Ce qu'on fait des défauts trouvés

**Séance 2.** La chaîne, vous la connaissez : lire, explorer, entraîner, soumettre. Aujourd'hui,
on s'occupe de ce que l'exploration a montré — les trous, les colonnes inutiles, les valeurs
impossibles — et on apprend à **décider**.

> **La règle :** une décision de nettoyage se **mesure**. Un effet nul est un résultat : il dit
> que cette décision-là ne se joue pas là.

**Livrable :** le **journal de nettoyage** (décision · choix · effet mesuré) et le **pipeline**
qui va de la table brute à la prédiction.


In [ ]:
# les outils
import glob
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
import matplotlib
import matplotlib.pyplot as plt


In [ ]:
# où sont les fichiers (Kahle ET en local) — on CHERCHE, on ne suppose pas un chemin
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie", os.path.join("..", "sortie"),
                            os.path.join("..", "Donnees_competition", "sortie")])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"

print("le dossier des données trouvé :", DOSSIER)


In [ ]:
# lire, et préparer la MÊME table de travail que la séance précédente
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"

A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La même préparation pour les deux tableaux."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X

X, Xt = matrice(train), matrice(test)
y = train[CIBLE]

print("les deux tableaux \n")
print("train :", train.shape, "(lignes, colonnes)")
print("test  :", test.shape, "(lignes, colonnes)")

print("la table de travail \n")
print("colonnes :", X.shape[1])
print("référence (classe majoritaire) :", round(100 * max(y.mean(), 1 - y.mean()), 2), "%")


### Le plafond — ce que les données permettent

La séance précédente a laissé une question ouverte : on a entraîné un modèle, il bat la référence,
et on pourrait en essayer d'autres. Jusqu'où ?
**Trois familles très différentes** — des arbres entraînés par corrections successives, un arbre
unique, un modèle linéaire — donnent ici **presque le même chiffre**. Ce n'est pas un hasard :
quand trois familles qui ne partagent pas leur mécanisme plafonnent au même endroit, le plafond
n'est pas dans le modèle. **Il est dans les données** — et le savoir évite de chercher pendant
six mois ce qui n'existe pas.


In [ ]:
# trois familles, la même table, la même mesure
Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

# la famille linéaire a besoin de nombres : indicatrices pour les catégorielles
# HistGradientBoosting prend les catégorielles telles quelles ; les deux autres familles
# veulent des nombres : on leur prépare des indicatrices — la leçon de la séance précédente.
CAT = [c for c in X.columns if str(X[c].dtype) == "category"]
# (le linéaire refuse aussi les trous : on les comble à 0, sans finesse — c'est une démo,#  pas une décision ; la vraie question du trou est la section suivante)
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xva_n = X_num.loc[Xtr.index], X_num.loc[Xva.index]

print("trois familles, la même table, la même mesure \n")
scores = {}
for nom, modele, donnees in (
        ("arbres (boosting)", HistGradientBoostingClassifier(random_state=0), (Xtr, Xva)),
        ("un arbre unique", DecisionTreeClassifier(random_state=0, max_depth=8), (Xtr_n, Xva_n)),
        ("modèle linéaire", LogisticRegression(solver="newton-cholesky"), (Xtr_n, Xva_n))):
    a, b = donnees
    scores[nom] = 100 * modele.fit(a, ytr).score(b, yva)
    print(f"  {nom:<22} {scores[nom]:6.2f} %")

print("\nl'écart entre la meilleure et la moins bonne famille :",
      round(max(scores.values()) - min(scores.values()), 2), "point(s).")
print("→ le plafond n'est pas dans le modèle.")


In [ ]:
# À vous — le plafond
# Le score ne bouge presque pas d'une famille à l'autre. La question qui suit est la bonne :
# « alors où sont les points qui restent ? »

# TODO : notez la ligne d'essai correspondante dans le tableau (même modèle, famille différente,
#        attendu « pareil », mesuré votre chiffre).


### Un trou n'est pas toujours un trou

Un trou dans une colonne n'a pas toujours la même valeur. Parfois la valeur manque **au hasard** :
le trou ne dit rien de la ligne, il dit seulement qu'on ne l'a pas notée. Parfois, au contraire,
**le fait qu'elle manque est en soi une information** — et le combler, c'est effacer cette
information.
La règle qui suit : **avant d'imputer, on regarde ce que le trou vaut.** C'est la décision la plus
fréquente du métier, et celle que les débutants prennent à l'envers — ils comblent d'abord, ils
regardent ensuite.


In [ ]:
# un trou ALÉATOIRE : quatre traitements, et ce que chacun change
def score_de(Xe, ye=None):
    """Entraîne et mesure, toujours sur la même découpe."""
    ye = y if ye is None else ye
    Xtr_m, Xva_m, ytr_m, yva_m = train_test_split(Xe, ye, test_size=0.2, stratify=ye, random_state=0)
    return 100 * HistGradientBoostingClassifier(random_state=0).fit(Xtr_m, ytr_m).score(Xva_m, yva_m)

colonne = "solde_credit"
mediane = X.copy()
mediane[colonne] = mediane[colonne].fillna(mediane[colonne].median())
avec_indicateur = mediane.copy()
avec_indicateur[colonne + "_etait_vide"] = X[colonne].isna().astype(int)
masque = X[colonne].notna()

print(f"la colonne {colonne}, quatre traitements du même trou \n")
print(f"  laissé tel quel            {score_de(X):6.2f} %")
print(f"  colonne retirée            {score_de(X.drop(columns=[colonne])):6.2f} %")
print(f"  trou comblé par la médiane {score_de(mediane):6.2f} %")
print(f"  médiane + indicateur       {score_de(avec_indicateur):6.2f} %")
print(f"  lignes à trou supprimées   {score_de(X[masque], ye=y[masque]):6.2f} %")
print("\n→ un trou aléatoire ne vaut rien : les cinq chiffres se tiennent à quelques centièmes.")


In [ ]:
# un trou STRUCTUREL : le même geste, un autre résultat
constantes = train["constantes_relevees"].fillna("").astype("category")

X_garde = X.copy()
X_garde["constantes_relevees"] = constantes

X_comble = X.copy()
X_comble["constantes_relevees"] = constantes.astype(str).replace("", "complet").astype("category")

print("la même colonne, trois traitements du même trou \n")
print(f"  le trou gardé comme catégorie  {score_de(X_garde):6.2f} %")
print(f"  le trou comblé par le mode     {score_de(X_comble):6.2f} %")
print(f"  la colonne retirée             {score_de(X):6.2f} %")

print("\n→ ici, garder le trou vaut plus que le combler. Le trou PORTE de l'information :")
print("  imputer sans regarder l'avait effacée.")


In [ ]:
# À vous — les trous
# Deux colonnes, deux trous, deux résultats. La différence entre elles est la leçon.

# TODO : pour chaque colonne à trous de la table, dites en une ligne si le trou est aléatoire
#        ou structuré, et pourquoi. Ecrivez le constat avec un chiffre.
#        (indice : un trou est structuré quand il est CORRÉLÉ à la cible)


### Les décisions qui ne se voient pas dans le score

Reste une famille de décisions, celles qu'on prend tous les jours sans se poser de question :
supprimer une colonne constante, écarter une colonne de bruit, traiter un code sans
documentation, retirer une ligne dupliquée, retirer une valeur impossible.
On les mesure **toutes**, et le résultat est presque toujours le même : **le score ne bouge pas.**
Ce n'est pas une déception, c'est un résultat. **Le nettoyage change rarement le score ; il change
ce que vous pouvez défendre**, et ce qui survit quand les données changent.


In [ ]:
# les colonnes inutiles : une constante, du bruit pur, deux redondantes
inutiles = [c for c in ("zone_clinique", "indice_meteo_jour", "score_territorial",
                        "age_mois", "cout_acte") if c in X.columns]

print("les colonnes qui n'apportent rien \n")
print("  toutes les colonnes      ", f"{score_de(X):6.2f} %")
print(f"  sans {len(inutiles)} colonnes inutiles", f"{score_de(X.drop(columns=inutiles)):6.2f} %")
print("\n→" , inutiles)


In [ ]:
# les catégories non documentées : un code arrive, personne ne sait ce qu'il vaut
if "code_assurance" in X.columns:
    inconnus = int(train["code_assurance"].isin([7, 9]).sum())
    print("les codes absents du dictionnaire publié \n")
    print("  lignes concernées :", inconnus, f"({100 * inconnus / len(train):.2f} %)")
    print("  colonne gardée    :", f"{score_de(X):6.2f} %")
    sans_code = score_de(X.drop(columns=["code_assurance"]))
    print("  colonne retirée   :", f"{sans_code:6.2f} %")
    print("\n→ la décision ne se joue pas sur le score, mais sur ce qu'on peut DIRE de ce code :")
    print("  le garder en sachant qu'il est inexpliqué, le rattacher, ou l'écarter — et pourquoi.")


In [ ]:
# les valeurs impossibles et les doublons
impossibles = train["age"] == -1
sans_impossibles = train[~impossibles]
sans_doublons = train.drop_duplicates()

print("rien touché                  ", f"{score_de(X):6.2f} %")
print("âges impossibles retirés     ",
      f"{score_de(matrice(sans_impossibles), ye=sans_impossibles[CIBLE]):6.2f} %")
print("lignes dupliquées retirées   ",
      f"{score_de(matrice(sans_doublons), ye=sans_doublons[CIBLE]):6.2f} %")
print("\n→ retirer des lignes peut même COÛTER un peu : on jette de l'information avec le défaut.")


In [ ]:
# À vous — les décisions
# Le score ne bouge pas. La question devient donc : qu'est-ce qu'on y gagne ?

# TODO : pour chaque décision ci-dessus, écrivez la raison qu'on donnerait à un audit —
#        en une phrase, sans invoquer le score.


### Le journal de nettoyage, et le pipeline

Chaque décision est maintenant mesurée. On les rassemble dans **un tableau** — le même esprit
que le tableau d'essais de la séance précédente, appliqué aux données : **décision · choix ·
effet mesuré**. C'est ce tableau qu'on rend, et c'est lui qui répond à la question d'un audit :
*pourquoi avez-vous fait ça ?*
Puis on assemble : une seule fonction qui va de la **table brute** à la **prédiction**, en portant
toutes les décisions. C'est le **pipeline** — et c'est ce que votre enseignant exécutera.


In [ ]:
# le journal de nettoyage — ce qui est lu n'est pas le score, c'est le tableau
JOURNAL = []

def journal(decision, choix, effet):
    """Une décision de nettoyage, le choix fait, et ce qu'il a rapporté EN POINTS."""
    JOURNAL.append({"décision": decision, "choix": choix, "effet mesuré": round(effet, 2)})
    print(pd.DataFrame(JOURNAL).to_string(index=False))

base = score_de(X)
journal("un trou aléatoire", "gardé tel quel", 0.0)
journal("un trou structurel", "gardé comme catégorie", score_de(X_garde) - base)
journal("des colonnes inutiles", "écartées", score_de(X.drop(columns=inutiles)) - base)
journal("une valeur impossible", "lignes retirées",
        score_de(matrice(sans_impossibles), ye=sans_impossibles[CIBLE]) - base)


In [ ]:
# le pipeline : de la table brute à la prédiction, en un seul endroit
def pipeline(brut):
    """Reçoit une table BRUTE (telle qu'elle arrive) et rend un tableau de 0/1."""
    d = matrice(brut)
    return modèle_final.predict(d).astype(int)

modèle_final = HistGradientBoostingClassifier(random_state=0).fit(X, y)
print("prédictions sur 500 lignes brutes :", pipeline(test.head(500)).shape)

soumission = pd.DataFrame({ID: test[ID], CIBLE: pipeline(test)})
soumission.to_csv("submission.csv", index=False)
print("le fichier écrit : submission.csv \n")
print("dimensions :", soumission.shape, "(lignes, colonnes)")


In [ ]:
# À vous — le pipeline
# Le pipeline porte TOUTES les décisions : c'est ce qui le rend défendable.

# TODO : ouvrez le fichier submission.csv et vérifiez qu'il a exactement autant de lignes
#        que test.csv. Puis dites, en une phrase, ce que votre pipeline ferait si une
#        colonne arrivait en texte alors qu'elle était numérique ici.


### Et une chose qu'on ne traite pas encore : le temps

Nos deux tableaux ne sont pas tirés au hasard dans une population : le fichier est un
**historique**, et le test vient **après** l'entraînement dans le temps. Personne ne nous l'a dit
— il faut le voir dans les dates.
**Pourquoi ça compte** : un découpage **au hasard** sur un historique laisse le futur entrer dans
l'apprentissage. Le score monte, et il ne voudra rien dire. On y revient à la séance de
validation : c'est là qu'on apprendra à découper sans se mentir.


In [ ]:
# la preuve, sans modèle : où s'arrête l'historique, où commence le test ?
prise, rdv = pd.to_datetime(train["date_rdv"]), pd.to_datetime(test["date_rdv"])
print("la période de l'historique \n")
print("  train : du", rdv.min().date(), "au", rdv.max().date())
print("  test  : du", prise.min().date(), "au", prise.max().date())
print("\n→ les deux périodes ne se chevauchent pas : le test vient après l'historique.")
print("  Un découpage au hasard mélangerait les deux — et le score ne voudrait plus rien dire.")


### Ce qu'on retient

- **Un plafond existe**, et il n'est pas dans le modèle : trois familles très différentes donnent
  ici presque le même chiffre. On le mesure avant de chercher à l'atteindre.
- **Un trou n'est pas toujours un trou** : aléatoire, il ne vaut rien ; structuré, il porte de
  l'information — donc **on mesure avant d'imputer**.
- **Le nettoyage change rarement le score.** Il change ce qu'on peut défendre, et ce qui survit
  quand les données changent.
- **Le journal de nettoyage** est le livrable : chaque décision, son choix, son effet mesuré.
